# 05.17 - Ensembling

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Ensembling combines multiple models to improve performance. We cover bagging, boosting, voting, and stacking.

## 2. Why Does This Matter?

Ensembles often beat any single model. Understanding how to combine diverse models is a key skill.

## 3. Prerequisites

- Units 05.6, 05.7 (Forests, Boosting)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain bagging, boosting, voting, stacking
- Build a voting ensemble
- Build a stacking ensemble
- Compare ensemble to single models

## 5. Mental Model

- **Bagging**: train models in parallel on bootstrap samples, average.
- **Boosting**: train sequentially, each corrects previous errors.
- **Voting**: combine predictions of diverse models.
- **Stacking**: train a meta-model on the predictions of base models.

Diversity is key - combining different models works best.


## 6. Generate Data

Use a classification dataset.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 700, Test: 300


## 7. Base Models

Train several diverse base models.


In [2]:
models = {
    'Logistic': LogisticRegression(max_iter=1000),
    'Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GB': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"{name:8s}: test accuracy={acc:.3f}")


Logistic: test accuracy=0.837
Tree    : test accuracy=0.833


D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM     : test accuracy=0.950


Forest  : test accuracy=0.917


GB      : test accuracy=0.907


## 8. Voting Ensemble

Combine predictions by majority vote.


In [3]:
voting = VotingClassifier(
    estimators=[('lr', models['Logistic']), ('tree', models['Tree']), ('svm', models['SVM'])],
    voting='soft'
)
voting.fit(X_train, y_train)
voting_acc = accuracy_score(y_test, voting.predict(X_test))
print(f"Voting ensemble accuracy: {voting_acc:.3f}")
print("\nVoting combines diverse models.")


Voting ensemble accuracy: 0.920

Voting combines diverse models.


D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 9. Stacking Ensemble

Train a meta-model on base model predictions.


In [4]:
stacking = StackingClassifier(
    estimators=[('lr', models['Logistic']), ('tree', models['Tree']), ('svm', models['SVM'])],
    final_estimator=LogisticRegression(max_iter=1000)
)
stacking.fit(X_train, y_train)
stacking_acc = accuracy_score(y_test, stacking.predict(X_test))
print(f"Stacking ensemble accuracy: {stacking_acc:.3f}")
print("\nStacking learns how to best combine base models.")


D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Stacking ensemble accuracy: 0.957

Stacking learns how to best combine base models.


D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\CODE\complete ml\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 10. Compare All

Summary of single models vs ensembles.


In [5]:
print(f"{'Model':<20} {'Accuracy':<10}")
print("-" * 32)
for name, m in models.items():
    print(f"{name:<20} {accuracy_score(y_test, m.predict(X_test)):<10.3f}")
print(f"{'Voting':<20} {voting_acc:<10.3f}")
print(f"{'Stacking':<20} {stacking_acc:<10.3f}")
print("\nEnsembles often outperform individual models.")


Model                Accuracy  
--------------------------------
Logistic             0.837     
Tree                 0.833     
SVM                  0.950     
Forest               0.917     
GB                   0.907     
Voting               0.920     
Stacking             0.957     

Ensembles often outperform individual models.


## 11. Failure Case: Homogeneous Ensemble

Combining identical models adds no diversity.


In [6]:
print("If all base models are identical, the ensemble adds nothing.")
print("Diversity is what makes ensembles work.")
print("\nUse different algorithms or different data subsets.")


If all base models are identical, the ensemble adds nothing.
Diversity is what makes ensembles work.

Use different algorithms or different data subsets.


## 12. Debugging: Common Errors

- **No diversity**: identical models.
- **Correlated errors**: models fail together.
- **Overfitting in stacking**: meta-model overfits.

## 13. Real-World Considerations

- Ensembles are more complex and slower.
- Use cross-validation for stacking.
- Diversity beats raw strength.

## 14. Common Mistakes

- Combining too many similar models.
- Not using CV in stacking.

## 15. When NOT to Use

- When interpretability is critical.
- When latency matters.

## 16. Challenge

Build a stacking ensemble with different base models and compare to a random forest.


In [7]:
# Challenge: stacking with more diverse models
stack2 = StackingClassifier(
    estimators=[('lr', LogisticRegression(max_iter=1000)), ('forest', RandomForestClassifier(n_estimators=100, random_state=42)), ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))],
    final_estimator=LogisticRegression(max_iter=1000)
)
stack2.fit(X_train, y_train)
acc2 = accuracy_score(y_test, stack2.predict(X_test))
print(f"Stacking (LR+Forest+GB): {acc2:.3f}")
print(f"Random forest alone:      {accuracy_score(y_test, models['Forest'].predict(X_test)):.3f}")
print("\nStacking diverse models can beat any single one.")


Stacking (LR+Forest+GB): 0.913
Random forest alone:      0.917

Stacking diverse models can beat any single one.


## 17. Closed-Book Recall

Without looking back:

1. What is the difference between bagging and boosting?
2. What is voting?
3. What is stacking?
4. Why is diversity important?

## 18. Teach-Back Questions

Explain to another person:

- The difference between voting and stacking.
- Why ensembles work.

## 19. Summary

You built voting and stacking ensembles and compared them to single models. Ensembles are a powerful way to boost performance.

## 20. Further Experiment

- Try weighted voting.
- Use a gradient boosting meta-model.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
